In [ ]:
#%pip install pandas
import pandas as pd
import re
import numpy as np
#%pip install plotly
#%pip install nbformat
import plotly.graph_objects as go
from input_data import segment_locations, years_to_check_combinations, exceptional_inds
# Load your dataset (replace with actual data)
data = pd.read_csv(r'’)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Create a DataFrame from the sample data
df = pd.DataFrame(data)
df['Amount'] = df['Amount'].str.replace(',', '').astype(float)
df['Amount'] = pd.to_numeric(df['Amount'])
print(df)


def filter_data(df, segment_location, years_to_check, exceptional_ind):
    # Filter rows based on specified conditions
    filtered_data = df[
        (df["Segment/Location"] == segment_location) &
        ((df["Year"] == years_to_check[0]) | (df["Year"] == years_to_check[1])) &
        (df["ExceptionalInd"] == exceptional_ind)
    ]
    print(filtered_data)
    # Identify unique cost types for 2021 and 2023
    gl_accounts_first_year = set(filtered_data[filtered_data["Year"] == years_to_check[0]]["GL Account"])
    gl_accounts_second_year = set(filtered_data[filtered_data["Year"] == years_to_check[1]]["GL Account"])
    # Find cost types that exist in 2021 but not in 2023 (and vice versa)
    gl_accounts_only_first_year = gl_accounts_first_year - gl_accounts_second_year
    #print(cost_types_only_2021)
    gl_accounts_only_second_year = gl_accounts_second_year - gl_accounts_first_year
    #print(cost_types_only_2023)
    print(f"GL Accounts that exist in the first year but not in the second year: {gl_accounts_only_first_year}")
    print(f"GL Accounts that exist in the second year but not in the first year: {gl_accounts_only_second_year}")
    
    filtered_data_cleaned = filtered_data[~filtered_data["GL Account"].isin(gl_accounts_only_first_year | gl_accounts_only_second_year)]
    print(filtered_data_cleaned)
    # Filter rows based on cost type conditions
    #filtered_data_cleaned = filtered_data[
       # ~(
        #    (filtered_data["Year"] == years_to_check[0])
        #    & (filtered_data["GL Account"].isin(cost_types_only_2021))
      #  )
      #  | ~(
      #      (filtered_data["Year"] == years_to_check[1])
     #       & (filtered_data["GL Account"].isin(cost_types_only_2023))
      #  )
   # ]

    return filtered_data_cleaned

# Specify the parameters
specific_segment_location = "7115 - Burlington GB"
specific_years_to_check = [2021, 2023]
specific_exceptional_ind = "NonExceptional_COGS - Non Exceptional COGS"


def calc_overall_costs(filtered_data_dynamic, years_to_check):
    # Ensure the "Amount" column is numeric
    #filtered_data_dynamic["Amount"] = pd.to_numeric(filtered_data_dynamic["Amount"], errors='coerce')

    # Define the start and end years
    start_year = years_to_check[0] 
    end_year = years_to_check[1]

    # Calculate overall cost for the first year
    overall_cost_first_year = filtered_data_dynamic[filtered_data_dynamic["Year"] == start_year]["Amount"].sum()

    # Calculate overall cost for the second year
    overall_cost_second_year = filtered_data_dynamic[filtered_data_dynamic["Year"] == end_year]["Amount"].sum()

    # Calculate change in cost
    change_in_cost = overall_cost_second_year - overall_cost_first_year

    # Print the results
    print(f"Overall cost in the first year: ${overall_cost_first_year:.2f}")
    print(f"Overall cost in the second year: ${overall_cost_second_year:.2f}")
    print(f"Change in cost from the first year to the second year: ${change_in_cost:.2f}")

    # Return the calculated values in case they need to be used later
    return overall_cost_first_year, overall_cost_second_year, change_in_cost

def calc_contributions(filtered_data_dynamic, years_to_check, change_in_cost):
    # Filter rows for the specified years
    df_first_year = filtered_data_dynamic[filtered_data_dynamic["Year"] == years_to_check[0]].rename(columns={"Amount": "Amount_First_Year"})
    df_second_year = filtered_data_dynamic[filtered_data_dynamic["Year"] == years_to_check[1]].rename(columns={"Amount": "Amount_Second_Year"})

    # Merge the DataFrames on "GL Account"
    merged_df = pd.merge(df_first_year, df_second_year, on="GL Account", how="outer")

    # Fill NaN values with 0
    merged_df.fillna(0, inplace=True)

    # Calculate the change in cost for each GL Account
    merged_df["Change"] = merged_df["Amount_Second_Year"] - merged_df["Amount_First_Year"]

    # Calculate the contribution for each GL Account
    merged_df["Contribution"] = merged_df["Change"] / change_in_cost

    # Sort the DataFrame by Contribution (highest to lowest)
    merged_df_sorted = merged_df.sort_values(by="Contribution", ascending=False)

    # Set the display format to avoid scientific notation
    pd.options.display.float_format = '{:.2f}'.format

    # Print the result
    print(merged_df_sorted[["GL Account", "Amount_First_Year", "Amount_Second_Year", "Change", "Contribution"]])

    return merged_df_sorted


# Create a dictionary to store contributions by GL Account
def create_contribution_dictionary(merged_df_sorted):
    contributions_by_gl = {row['GL Account']: row['Contribution'] for index, row in merged_df_sorted.iterrows()}

    # Print the contributions_by_gl dictionary
    for account, contribution in contributions_by_gl.items():
        print(f"{account}: {contribution:.2f}")
    return contributions_by_gl

def waterfall_trace_each_contribution(merged_df_sorted):
    # Create custom colors: red for increases, green for decreases
    merged_df_sorted["Color"] = np.where(merged_df_sorted["Change"] > 0, 'red', 'green')

    # Create a waterfall trace
    trace = go.Waterfall(
        x=merged_df_sorted["GL Account"],
        y=merged_df_sorted["Contribution"],
        measure=['relative'] * len(merged_df_sorted),
        textposition="outside",
        text=[f"${change:.2f} | {val:.1%}" for val, change in zip(merged_df_sorted["Contribution"], merged_df_sorted["Change"])],
        connector={"line": {"color": "rgb(63, 63, 63)"}},
        increasing=dict(marker=dict(color="red")),
        decreasing=dict(marker=dict(color="green")),
    )

    # Create layout
    layout = go.Layout(
        title="Waterfall Diagram of Contributions",
        xaxis=dict(title="GL Account"),
        yaxis=dict(title="Contribution", range=[0, 2]),
        showlegend=True
    )

    # Create figure
    fig = go.Figure(data=[trace], layout=layout)

    # Add custom legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color='red'),
                             legendgroup='Increase',
                             showlegend=True,
                             name='Increase'))

    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color='green'),
                             legendgroup='Decrease',
                             showlegend=True,
                             name='Decrease'))

    # Show the plot
    fig.show()


def create_category_dictionary(merged_df, change_in_cost):
    category_name_mapping = {
        '60_85': 'Labor',
        '62': 'Utilities/Fuel', 
        '63': 'Repair',
        '64': 'General/Rental',
        '65': 'Administrative/Operational',
        '66': 'Operational/Environmental',
        '67': 'Marketing',
        '68': 'Development/Legal/Misc',
        '50': 'Propane',
        '51': 'Freight'
    }
    
    # Create a dictionary to store contributions by category
    contributions_by_category = {}
    category_details = {}
    total_change_by_category = {}
    # Iterate over rows in the DataFrame
    for index, row in merged_df.iterrows():
        # Extract the first two digits of the account code
        category = row['GL Account'][:2]
    
    # Combine categories 60 and 85
        if category in ['60', '85']:
            category = '60_85'
    
        # Add the contribution to the category
        contributions_by_category.setdefault(category, 0)
        contributions_by_category[category] += row['Change']

        # Store the details for each GL Account in the category
        category_details.setdefault(category, [])
        category_details[category].append(row['GL Account'])

        # Calculate the total change for each category
        total_change_by_category.setdefault(category, 0)
        total_change_by_category[category] += row['Change']

    # Calculate the percentage contribution for each category
    for category in contributions_by_category:
        contributions_by_category[category] /= change_in_cost

    # Sort the dictionary by contribution (highest to lowest)
    contributions_by_category_sorted = dict(sorted(contributions_by_category.items(), key=lambda item: item[1], reverse=True))

    # Set the display format to avoid scientific notation
    pd.options.display.float_format = '{:.2f}'.format

    # Convert the sorted dictionary to a DataFrame for display
    contributions_df = pd.DataFrame(list(contributions_by_category_sorted.items()), columns=['Category', 'Contribution'])

    # Add the total change column to the DataFrame
    contributions_df['Total Change'] = contributions_df['Category'].map(total_change_by_category)

    # Print the result
    print(contributions_df)

    # Print the details for each category
    for category, gl_accounts in category_details.items():
        category_name = category_name_mapping.get(category, category)
        print(f"\nGL Accounts in {category_name}:")
        print(", ".join(gl_accounts))

    # Return the dictionaries
    return contributions_by_category_sorted, category_details, total_change_by_category


def waterfall_trace_categories(contributions_by_category_sorted, total_change_by_category):
    category_name_mapping = {
        '60_85': 'Labor',
        '62': 'Utilities/Fuel', 
        '63': 'Repair',
        '64': 'General/Rental',
        '65': 'Administrative/Operational',
        '66': 'Operational/Environmental',
        '67': 'Marketing',
        '68': 'Development/Legal/Misc',
        '50': 'Propane',
        '51': 'Freight'
    }
    
    # Prepare the data
    categories = list(contributions_by_category_sorted.keys())
    contributions = list(contributions_by_category_sorted.values())
    changes = [total_change_by_category[k] for k in categories]

    # Map category codes to their names
    category_names = [category_name_mapping.get(cat, cat) for cat in categories]

    # Create DataFrame for easier handling
    df = pd.DataFrame({
        "Category": category_names,
        "Contribution": contributions,
        "Change": changes
    })

    # Create custom colors: red for increases, green for decreases
    df["Color"] = np.where(df["Change"] > 0, 'red', 'green')

    # Create a waterfall trace
    trace = go.Waterfall(
        x=df["Category"],
        y=df["Contribution"],
        measure=['relative'] * len(df),
        textposition="outside",
        text=[f"${change:.2f} | {val:.1%}" for val, change in zip(df["Contribution"], df["Change"])],
        connector={"line": {"color": "rgb(63, 63, 63)"}},
        increasing=dict(marker=dict(color="red")),
        decreasing=dict(marker=dict(color="green")),
    )

    # Create layout
    layout = go.Layout(
        title="Waterfall Diagram of Contributions by Category",
        xaxis=dict(title="Category"),
        yaxis=dict(title="Contribution", range=[0, 2]),
        showlegend=True
    )

    # Create figure
    fig = go.Figure(data=[trace], layout=layout)

    # Add custom legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color='red'),
                             legendgroup='Increase',
                             showlegend=True,
                             name='Increase'))

    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10, color='green'),
                             legendgroup='Decrease',
                             showlegend=True,
                             name='Decrease'))

    # Show the plot
    fig.show()

# Define the report type
report_type = "All"  # or "All"

# Define specific segment, year, and indicator to match one combination in the loop if report_type is "Single"
specific_segment_location = "7103 - Fort Dodge GB"
specific_years_to_check = [2021, 2023]
specific_exceptional_ind = "NonExceptional_COGS - Non Exceptional COGS"

if report_type == "Single":
    # Run for the specific combination only
    filtered_data_dynamic = filter_data(df, specific_segment_location, specific_years_to_check, specific_exceptional_ind)
    print(f"Filtered data for {specific_segment_location} and years {specific_years_to_check} and exceptional indicator {specific_exceptional_ind}:")
    print(filtered_data_dynamic)
    
    # Calculate overall costs
    overall_cost_first_year, overall_cost_second_year, change_in_cost = calc_overall_costs(filtered_data_dynamic, specific_years_to_check)
    
    # Calculate contributions
    merged_df_sorted = calc_contributions(filtered_data_dynamic, specific_years_to_check, change_in_cost)
    
    # Create contribution dictionary
    contributions_by_gl = create_contribution_dictionary(merged_df_sorted)
    
    # Waterfall trace for each contribution
    waterfall_trace_each_contribution(merged_df_sorted)
    
    # Create category dictionary
    contributions_by_category_sorted, category_details, total_change_by_category = create_category_dictionary(merged_df_sorted, change_in_cost)
    
    # Waterfall trace for categories
    waterfall_trace_categories(contributions_by_category_sorted, total_change_by_category)

elif report_type == "All":
    # Run for all combinations
    for segment_location in segment_locations:
        for years_to_check in years_to_check_combinations:
            for exceptional_ind in exceptional_inds:
                filtered_data_dynamic = filter_data(df, segment_location, years_to_check, exceptional_ind)
                print(f"Filtered data for {segment_location} and years {years_to_check} and exceptional indicator {exceptional_ind}:")
                print(filtered_data_dynamic)
                
                # Calculate overall costs
                overall_cost_first_year, overall_cost_second_year, change_in_cost = calc_overall_costs(filtered_data_dynamic, years_to_check)
                
                # Calculate contributions
                merged_df_sorted = calc_contributions(filtered_data_dynamic, years_to_check, change_in_cost)
                
                # Create contribution dictionary
                contributions_by_gl = create_contribution_dictionary(merged_df_sorted)
                
                # Waterfall trace for each contribution
                waterfall_trace_each_contribution(merged_df_sorted)
                
                # Create category dictionary
                contributions_by_category_sorted, category_details, total_change_by_category = create_category_dictionary(merged_df_sorted, change_in_cost)
                
                # Waterfall trace for categories
                waterfall_trace_categories(contributions_by_category_sorted, total_change_by_category)
